In [1]:
import pandas as pd
import numpy as np
import json
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import SVG, display
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit import DataStructs
from rdkit.Chem import MACCSkeys
from rdkit.Chem import AllChem
import pandas as pd
import rdchiral
import itertools
import pickle
import sys
from tqdm import tqdm 
from rdchiral.main import rdchiralRun, rdchiralRunText, rdchiralReaction, rdchiralReactants
import cobra
from itertools import chain, combinations
import sys
import multiprocessing
from functools import partial
import matplotlib.pyplot as plt
import multiprocessing
from tqdm import tqdm
sys.path.append('../')

from common import *

### input and output

In [2]:
# step1
#input
mnxreac_smile_atom_mapping_rules_file_path = './merged_atommap_rules.csv'
target_drug_path = '../../Data/Drug/drugs_metabolites_db_v5.1.12.xlsx'
###output
result_all_file = "../../Results/drugbank_all_result_newrules.csv"

In [3]:
def process_rule(rule_smiles, mets_total_smiles):
    score = []
    smiles = []
    if 'C' in rule_smiles or 'c' in rule_smiles:
        for j in mets_total_smiles:
            current_score = calculate_similarity(rule_smiles, j)
            if current_score > 0.3:               ########################################################################
                score.append(current_score)
                smiles.append(j)
    else:
        smiles.append(rule_smiles)
        score.append(1)

    if len(score) == 0:
        return rule_smiles, [], []
    elif len(score) > 50:
        top40_data = sorted(zip(score, smiles), reverse=True)[:50]
    else:
        top40_data = sorted(zip(score, smiles), reverse=True)
    
    top40_scores, top40_smiles = zip(*top40_data)
    
    return rule_smiles, list(top40_smiles), list(top40_scores)

####################################################################################

def count_carbons(smiles):
    """计算一个 SMILES 字符串中碳 (C) 的个数"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    return sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')


def process_rule(rule_smiles, mets_total_smiles):
    score = []
    smiles = []
    
    # 计算 rule_smiles 的碳原子数
    rule_c_count = count_carbons(rule_smiles)

    if 'C' in rule_smiles or 'c' in rule_smiles:
        for j in mets_total_smiles:
            # 计算当前代谢物的碳原子数
            j_c_count = count_carbons(j)
            
            # 计算相似度
            current_score = calculate_similarity(rule_smiles, j)
            
            # 计算罚分
            if rule_c_count!=0:
                if rule_c_count > j_c_count:
                    penalty = (rule_c_count - j_c_count) / (rule_c_count) /10.216
                elif rule_c_count < j_c_count:
                    penalty = (j_c_count - rule_c_count) / (rule_c_count) /8.828
                else:
                    penalty = 0
            else:
                penalty = 0
            
            # 应用罚分
            adjusted_score = max(0, current_score - penalty)

            # 仅保留相似度大于0.3的结果
            if adjusted_score > -100:
                score.append(adjusted_score)
                smiles.append(j)
    
    else:
        smiles.append(rule_smiles)
        score.append(1)

    if len(score) == 0:
        return rule_smiles, [], []
    # elif len(score) > 100:
    #     top20_data = sorted(zip(score, smiles), reverse=True)[:100]
    else:
        top20_data = sorted(zip(score, smiles), reverse=True)
    
    top20_scores, top20_smiles = zip(*top20_data)
    
    return rule_smiles, list(top20_smiles), list(top20_scores)

In [4]:
def process_retrorule(index_row, mets_total_smiles,retrosys_smiles_calculate_similarity_pd,heterologous_met_smiles):
    rxndb_list_tmp = []
    # newdbSmiles_tmp = []
    failedrxn_tmp = []
    index, row = index_row
    rxn_id, ECnumber, reactants, products = row['MNX_ID'], row['classifs'], row['substrate_smiles'], row['product_smiles']
    deprecated_equ_smiles, reactantSMARTs, productSMARTs, rule = row['deprecated_equ_smiles'], row['ReactantsSMARTs'], row['ProductSMARTs'], row['RetroRules']
    deprecated_equ_smiles = deprecated_equ_smiles.split('>>')
    if rxn_id not in ['MNXR101882', 'MNXR101884', 'MNXR101879', 'MNXR101885', 'MNXR101886', 'MNXR101887',
                        'MNXR103528', 'MNXR103575']:  # ATP NADH conversion stuff
        productSMARTs = productSMARTs.split('.')
        products = products.split('.')
        new_products_smiles = []
        scores = []
        
        if len(productSMARTs) == len(products):
            for x in range(len(productSMARTs)):  # prepare mets with substructure and sort the reactants combination
                new_products_smile_one,score_one = get_calculate_similarity(products[x],retrosys_smiles_calculate_similarity_pd)
                new_products_smiles.append(new_products_smile_one)
                scores.append(score_one)

            product_prepared = []
            scores_prepared = []
            for lst, score_lst in zip(new_products_smiles, scores):
                if heterologous_met_smiles in lst:
                    if len(new_products_smiles) == 1:  # if only one sub list
                        product_prepared.append([heterologous_met_smiles])
                        scores_prepared.append(score_lst)
                    else:
                        other_lists = [sub_lst for sub_lst in new_products_smiles if sub_lst != lst]
                        tmp_product = list(itertools.product([heterologous_met_smiles], *other_lists))
                        product_prepared += tmp_product

                        tmp_scores = list(itertools.product([score_lst[new_products_smiles.index(lst)]], *scores))
                        scores_prepared += tmp_scores
            # print('product_prepared',product_prepared,len(product_prepared))
            for m in range(0, len(product_prepared)):  # apply the rule and output the rxn
                smile = product_prepared[m]
                score = scores_prepared[m]
                
                if len(smile) > 1:
                    product_smile = '.'.join(smile)
                else:
                    product_smile = smile[0]

                retrorule_tmp = rule.split('>>')
                retrorule_tmp = '(' + retrorule_tmp[0] + ')>>' + '(' + retrorule_tmp[1] + ')'            
                product_smile_tmp = product_smile
                # print('retrorule_tmp',retrorule_tmp)
                # print('product_smile',product_smile)
                try:
                    # print('product_smile_tmp',product_smile_tmp)
                    product_mol = Chem.MolFromSmiles(product_smile_tmp)
                    reaction = AllChem.ReactionFromSmarts(retrorule_tmp)
                    reactant_smiles = reaction.RunReactants((product_mol,))
                    smiles = [Chem.MolToSmiles(x[0]) for x in reactant_smiles]
                    reactant_smiles = list(set(smiles))

                    for reactant_smile in reactant_smiles:
                        if len(deprecated_equ_smiles) > 0:
                            if deprecated_equ_smiles[0] == '':
                                reactant_smile_final = reactant_smile
                            else:
                                reactant_smile_final = reactant_smile + '.' + deprecated_equ_smiles[0]
                            if deprecated_equ_smiles[1] == '':
                                product_smile_final = product_smile_tmp
                            else:
                                product_smile_final = product_smile_tmp + '.' + deprecated_equ_smiles[1]
                        rxn_smiles = reactant_smile + '>>' + product_smile_tmp
                        rxn_smiles_final = reactant_smile_final + '>>' + product_smile_final
                        
                        tmp = {
                            'EC number': ECnumber,
                            'rule': rule,
                            'templateID': rxn_id,
                            'templateSubstrate': products,
                            'rxn_smiles_basic': rxn_smiles,
                            'rxn_smiles_final': rxn_smiles_final,
                            'reactant_smile': reactant_smile,
                            'productsmile': product_smile_tmp,
                            'similarity': score
                        }
                        if check_rule(tmp['rxn_smiles_basic'])=='blance':
                            rxndb_list_tmp.append(tmp)  
                            # mets_tmp = reactant_smile.split('.')      
                            # for reactant in mets_tmp:             
                            #     if reactant not in mets_total_smiles and reactant not in newdbSmiles_tmp:         
                            #         newdbSmiles_tmp.append(reactant)        
                except:
                    failedrxn_tmp.append(rxn_id)
                
    return(rxndb_list_tmp,failedrxn_tmp)

In [5]:
def process_ec_list(ec_list,topn):
    processed_set = set()
    
    for ec in ec_list:
        for sub_ec in ec.split(';'):
            parts = sub_ec.strip().split('.')
            if len(parts) >= 3:  # Only process EC numbers with at least 3 parts
                processed_set.add('.'.join(parts[:topn]))
    
    return list(processed_set)

# Example usage
input_ec_numbers = ['4.1.3.-', '3.1.3.1', '1.1.3.100;2.1.4.2']
output_ec_numbers = process_ec_list(input_ec_numbers,3)
print(output_ec_numbers)

['2.1.4', '4.1.3', '3.1.3', '1.1.3']


In [6]:
def check_productsmile(smiles):#only keep smiles with carbon
    smiles_lst = smiles.split('.')
    smiles_lst = [x for x in smiles_lst if smiles_has_carbon(x)]
    return len(smiles_lst)

In [7]:
rules_product_smiles_lst = []
retrorules = pd.read_csv(mnxreac_smile_atom_mapping_rules_file_path)
# retrorules = retrorules[retrorules['MNX_ID'].str.contains('MNXR97814')]
for index,row in retrorules.iterrows():
    tmp  = row['product_smiles'].split('.')
    for i in tmp:
        if i not in rules_product_smiles_lst:
            rules_product_smiles_lst.append(i)
print('rules_product_smiles_lst',len(rules_product_smiles_lst))


/home/wuke/anaconda3/envs/AIGEM/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3457: DtypeWarning: Columns (1,2,4,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


rules_product_smiles_lst 14792


In [8]:
target_drug = pd.read_excel(target_drug_path)
target_drug.rename(columns={'metabolites_smiles':'product_smiles'},inplace=True)
# target_drug = target_drug[target_drug['drug_name']=='digitoxin']
# target_drug = target_drug.loc[0:2,:]
# target_drug = target_drug.iloc[error_index[300:],:].reset_index(drop=True)
target_drug = target_drug[target_drug['drug_smiles'].apply(lambda x:normalize_smiles(x)!=None)]
target_drug = target_drug[['drugbank_id','drug_name','drug_smiles','product_smiles','our_predicitions','MicrobeRX_predicitions']]
target_drug

,drugbank_id,drug_name,drug_smiles,product_smiles,our_predicitions,MicrobeRX_predicitions
0,DB00091,Cyclosporine,CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\C=C\C)N...,CC[C@@H]1NC(=O)[C@H]([C@H](O)[C@H](C)C\C=C\C)N...,NaN,NaN
1,DB01120,Gliclazide,[H][C@@]12CCC[C@]1([H])CN(C2)NC(=O)NS(=O)(=O)C...,[H][C@@]12CCC[C@]1([H])CN(C2)N=C(O)NS(=O)(=O)C...,NaN,yes
2,DB01132,Pioglitazone,CCC1=CN=C(CCOC2=CC=C(CC3SC(=O)NC3=O)C=C2)C=C1,OC1=CC=C(CC2SC(=O)NC2=O)C=C1,NaN,yes
3,DB01131,Proguanil,CC(C)NC(=N)NC(=N)NC1=CC=C(Cl)C=C1,NC(=N)NC(=N)NC1=CC=C(Cl)C=C1,NaN,yes
4,DB01131,Proguanil,CC(C)NC(=N)NC(=N)NC1=CC=C(Cl)C=C1,CC1(C)N=C(N)N=C(N)N1C1=CC=C(Cl)C=C1,NaN,yes
...,...,...,...,...,...,...
1269,DB00643,Mebendazole,COC(=O)NC1=NC2=C(N1)C=C(C=C2)C(=O)C1=CC=CC=C1,N=C1NC2=C(N1)C=C(C=C2)C(=O)C1=CC=CC=C1,NaN,NaN
1270,DB00641,Simvastatin,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,[H][C@@](O)(CC[C@@]1([H])[C@@H](C)C=CC2=C[C@H]...,NaN,yes
1271,DB00641,Simvastatin,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,[H]O[C@@]1([H])C([H])([H])C(=O)O[C@]([H])(C([H...,NaN,NaN
1272,DB00641,Simvastatin,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,[H]O[C@@]1([H])C([H])([H])C(=O)O[C@]([H])(C([H...,NaN,yes


In [9]:
# # 读取gut clean ec预测结果
# with open('../../Results/AGORA2_strain_eclist_top3_common.pkl', 'rb') as f:
#     loaded_top3 = pickle.load(f)

# with open('../../Results/AGORA2_strain_eclist_top4_common.pkl', 'rb') as f:
#     loaded_top4 = pickle.load(f)

# print("读取完成！")
# print(f"Top3 列表长度：{len(loaded_top3)}")
# print(f"Top4 列表长度：{len(loaded_top4)}")

## byproduct prediction

In [10]:
result_num = {'drug_name':[],
              'reaction_num':[],
              'ec4_num':[],
              'ec4_list':[],
              'ec3_num':[],
              'ec3_list':[],
              'success_predict':[],
              'success_ecnumber':[],
              }
# filter_info = {'drug_carbon':[],
#                'success_carbon':[],
#                'error_carbon':[],
#                'success_sim_score':[],
#                'error_sim_score':[],
#                }
# 循环开始前，删掉旧文件
if os.path.exists(result_all_file):
    os.remove(result_all_file)
# target_drug = target_drug.iloc[1140:,:]
for index,row in tqdm(target_drug.iterrows(),total=len(target_drug)):
    print(row['drug_name'])
    heterologous_met_smiles = row['drug_smiles']
    # heterologous_met_smiles = 'FC1=CNC(=O)NC1=O'
    # heterologous_met_smiles = 'N[C@@H](CC1=CC(O)=C(O)C=C1)C(O)=O'
    # heterologous_met_smiles = '[NH3+][C@@H](Cc1ccc(O)c(O)c1)C(=O)[O-]' ######## L-dopa
    heterologous_met_smiles = normalize_smiles(heterologous_met_smiles)
    # model_met = target_drug['smiles'].to_list()
    model_met = [heterologous_met_smiles,heterologous_met_smiles,heterologous_met_smiles,'O=C([O-])CCC(=O)C(=O)[O-]',
                 'C[S+](CC[C@@H](C(=O)[O-])N)C[C@@H]1[C@H]([C@H]([C@@H](O1)N2C=NC3=C(N=CN=C32)N)O)O',
                 'C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)CSCC[C@@H](C(=O)O)N)O)O)N',
                 'O=C(O)[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@H]1O',# β-D-葡萄糖醛酸
                 'CC(=O)SCCNC(=O)CCNC(=O)[C@@H](C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)N2C=NC3=C(N=CN=C32)N)O)OP(=O)(O)O)O', # Acetyl-CoA
                 ]
    new_met = []

    # add heterologous_met_smiles to model_met list
    new_model_and_heterologous_met = model_met + new_met
    new_model_and_heterologous_met.append(heterologous_met_smiles)
    new_model_and_heterologous_met = [smile for smile in new_model_and_heterologous_met if '.' not in smile]
    new_model_and_heterologous_met = [smile for smile in new_model_and_heterologous_met if 'C' in smile or 'c' in smile] ###只筛第一轮
    # print(len(new_model_and_heterologous_met))
    #calculate similarity
    num_processes = 100
    pool = multiprocessing.Pool(num_processes)
    retrosys_smiles_calculate_similarity = {'rules_smiles': [],
                                            'smiles_in_mets_total_smiles': [],
                                            'scores': []}


    process_rule_partial = partial(process_rule, mets_total_smiles=new_model_and_heterologous_met)
    for result in tqdm(pool.imap(process_rule_partial, rules_product_smiles_lst, chunksize=200), total=len(rules_product_smiles_lst)):
        if result[0] not in retrosys_smiles_calculate_similarity['rules_smiles']:
            retrosys_smiles_calculate_similarity['rules_smiles'].append(result[0]) 
            retrosys_smiles_calculate_similarity['smiles_in_mets_total_smiles'].append(result[1]) 
            retrosys_smiles_calculate_similarity['scores'].append(result[2]) 
    pool.close()
    pool.join()

    retrosys_smiles_calculate_similarity_pd = pd.DataFrame(retrosys_smiles_calculate_similarity)
##############################################
    # for index4, row4 in retrosys_smiles_calculate_similarity_pd.iterrows():
    #     if row4['rules_smiles'] not in row4['smiles_in_mets_total_smiles']:
    #         # 修改 DataFrame 中嵌套的列表
    #         retrosys_smiles_calculate_similarity_pd.at[index4, 'smiles_in_mets_total_smiles'].append(row4['rules_smiles'])
##############################################
    rxndb_list = [] 
    # newdbSmiles = []
    failedrxn = []
    num_processes =100  #  adjust the number of processes according to your needs
    pool = multiprocessing.Pool(num_processes)
    # Execute in parallel using process pool

    # Create a partial function that passes mets_total_smiles and mets_total_mol as default parameters to the process_retrorule function
    process_retrorule_partial = partial(process_retrorule, mets_total_smiles=new_model_and_heterologous_met,retrosys_smiles_calculate_similarity_pd = retrosys_smiles_calculate_similarity_pd,heterologous_met_smiles = heterologous_met_smiles)

    # Using the process pool to execute in parallel, the chunksize parameter is placed inside the imap function.
    for result in tqdm(pool.imap(process_retrorule_partial, retrorules.iterrows(), chunksize=200), total=len(retrorules)):
        # if result == ([], [], []):
        #     pass
        # else:
        #     print(result)
        if result is not None:
            if result[0] :  #  check if the result is not empty
                rxndb_list.extend(result[0]) 
            # if result[1] is not None:
            #     newdbSmiles.extend(result[1])
            if result[1]:
                failedrxn.extend(result[1])  

    pool.close()
    pool.join()

    rxndb = {}
    for i, rxn_dict in enumerate(rxndb_list):
        # generate a unique key for each reaction, for example "rxn0"，"rxn1"，"rxn2"，...
        key = f'rxn{i+1}'
        # merge to the dictionary
        rxndb[key] = rxn_dict

    # print('rxndb num',len(rxndb))
    # print('failedrxn num',len(failedrxn))

    ecnumber_list = []
    for _,x in rxndb.items():
        ec_list = x['EC number'].split(';')
        ec_list = process_ec_list(ec_list,3)
        ecnumber_list += ec_list

    ecnumber_list = list(set(ecnumber_list))   
    result_df = {'ID':[],
            'reactant_smile':[],
            'productsmile':[],
            'rxn_smiles_basic':[],
            'templateID':[],
            'EC number':[],}

    for k,v in rxndb.items():
        result_df['ID'].append(k)
        result_df['reactant_smile'].append(v['reactant_smile'])
        result_df['productsmile'].append(v['productsmile'])
        result_df['rxn_smiles_basic'].append(v['rxn_smiles_basic'])
        result_df['templateID'].append(v['templateID'])
        result_df['EC number'].append(v['EC number'])

    result_df = pd.DataFrame({'ID': result_df['ID'],
                        'reactant_smile': result_df['reactant_smile'],
                        'productsmile': result_df['productsmile'],
                        'rxn_smiles_basic': result_df['rxn_smiles_basic'],
                        'templateID': result_df['templateID'],
                        'EC number': result_df['EC number'],
                        })

    result_df.drop_duplicates(subset=['reactant_smile', 'productsmile'], inplace=True)
    result_df.reset_index(drop=True, inplace=True)

    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.OO']
    result_df = result_df[result_df['productsmile']!='OO.'+ heterologous_met_smiles]
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=C=O']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=S([O-])[O-]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[SH-]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.O=P([O-])([O-])OP(=O)([O-])O']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.II']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[NH4+]']
    result_df = result_df[result_df['productsmile']!=heterologous_met_smiles+'.[Br-]']
    
    try:
        res = result_df['reactant_smile'].to_list()
        ec_lst = []
        for k,v in rxndb.items():
            if v['reactant_smile'] in res:
                ec_lst.append(v['EC number'])
    except:
        ec_lst = []
        pass
    # print('result_df',len(result_df))
    # print(ec_lst)
    ec_lst_4 = process_ec_list(ec_lst,4)
    ec_lst_3 = process_ec_list(ec_lst,3)
    # print(len(ec_lst_3))
    # print(len(ec_lst_4))
    
    result_num['drug_name'].append(row['drug_name'])
    result_num['reaction_num'].append(len(result_df))
    result_num['ec4_num'].append(len(ec_lst_4))
    result_num['ec4_list'].append(str(ec_lst_4))
    result_num['ec3_num'].append(len(ec_lst_3))
    result_num['ec3_list'].append(str(ec_lst_3))
    
    success_predict = 0
    success_ecnumber = []
    success_ID = []
    if row['product_smiles'] != 'unknown' and row['drug_name'] != 'Lactulose':
        target_smiles = row['product_smiles'].split('.')[0]
        for index2,row2 in result_df.iterrows():
            smiles_list = row2['reactant_smile'].split('.')
            if any(compare_smiles_inchikey(smiles,target_smiles) for smiles in smiles_list):
                success_predict =1
                print(1)
                success_ID.append(row2['ID'])
                tmp_ec_lst = row2['EC number'].split(';')
                success_ecnumber+=process_ec_list(tmp_ec_lst,3)
    
    result_num['success_predict'].append(success_predict)
    result_num['success_ecnumber'].append(str(list(set(success_ecnumber))))

  0%|          | 0/1274 [00:00<?, ?it/s]

Cyclosporine


100%|██████████| 34899/34899 [00:14<00:00, 2480.67it/s]


1
1


  0%|          | 1/1274 [00:30<10:43:07, 30.31s/it]

...

1
Dexmethylphenidate


100%|██████████| 34899/34899 [00:10<00:00, 3363.64it/s]


1
1
1
1


100%|██████████| 1274/1274 [7:58:28<00:00, 22.53s/it]

1
1


In [11]:
result_num_df = pd.DataFrame(result_num)
result_num_df[result_num_df['success_predict']==1]
# failed_index2 = result_num_df[result_num_df['success_predict'] == 0].index
# print(failed_index2)

,drug_name,reaction_num,ec4_num,ec4_list,ec3_num,ec3_list,success_predict,success_ecnumber
0,Cyclosporine,1342,560,"['2.3.1.285', '3.2.1.3', '3.2.1.20', '6.3.1.10...",63,"['1.14.14', '3.1.1', '1.13.12', '1.1.99', '2.5...",1,"['1.14.15', '1.14.-']"
1,Gliclazide,186,174,"['1.14.13.71', '1.14.13.44', '3.2.1.137', '1.1...",41,"['1.14.14', '3.1.1', '1.14.-', '5.1.3', '1.3.1...",1,['1.14.14']
2,Pioglitazone,287,235,"['2.1.1.M10', '6.3.1.10', '1.14.13.71', '1.14....",54,"['1.8.3', '1.14.14', '1.13.12', '3.1.1', '1.14...",1,['1.14.14']
4,Proguanil,111,124,"['1.14.13.71', '1.14.13.44', '1.14.11.-', '1.1...",28,"['1.14.14', '1.14.-', '1.11.1', '1.14.20', '1....",1,[]
6,Tolbutamide,205,193,"['1.14.13.71', '1.14.13.44', '3.2.1.137', '1.1...",44,"['1.14.14', '1.13.12', '3.1.1', '1.14.-', '5.1...",1,['1.14.14']
...,...,...,...,...,...,...,...,...
1269,Mebendazole,311,214,"['2.1.1.M10', '1.14.13.71', '1.14.13.44', '3.2...",45,"['1.14.14', '3.1.1', '1.13.12', '1.14.-', '5.1...",1,['3.1.1']
1270,Simvastatin,539,545,"['2.3.1.285', '3.2.1.3', '3.2.1.20', '1.1.1.28...",59,"['1.14.14', '3.1.1', '1.13.12', '1.1.99', '2.5...",1,"['2.3.1', '3.1.1', '3.4.21']"
1271,Simvastatin,539,545,"['2.3.1.285', '3.2.1.3', '3.2.1.20', '1.1.1.28...",59,"['1.14.14', '3.1.1', '1.13.12', '1.1.99', '2.5...",1,['1.3.1']
1272,Simvastatin,539,545,"['2.3.1.285', '3.2.1.3', '3.2.1.20', '1.1.1.28...",59,"['1.14.14', '3.1.1', '1.13.12', '1.1.99', '2.5...",1,[]


In [19]:
result_num_df['success_predict'].value_counts()

1    944
0    330
Name: success_predict, dtype: int64